# RASCALS_IBM Z Datathon 2026 Project
## ContinuityGuard: 03 - Engineering audit and platform proof

ANBU SELVAN T; SAHANA G; DHARSHINI E; POOJA V

Hindustan Institute of Technology and Science

This notebook tests the hardened source on LinuxONE, exercises actual local authenticated HTTP ingestion, and verifies native Granite evidence generation. Synthetic fixtures are explicitly labeled. It does not turn a reproducibility rerun into a new independent model evaluation.

In [1]:
from pathlib import Path
import os, sys, platform, json
ROOT = Path.cwd()
if not (ROOT / 'continuityguard').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'continuityguard').is_dir(), 'Open this notebook from the project notebooks folder'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('Team: RASCALS')
print('Architecture:', platform.machine())
print('Python:', platform.python_version())
print('Project:', ROOT)
assert platform.machine() == 's390x', 'This submission notebook is intended to run on LinuxONE'


Team: RASCALS
Architecture: s390x
Python: 3.12.15
Project: /home/ibm-user/shared/continuityguard


In [2]:
import unittest, io, time, importlib.metadata
suite = unittest.defaultTestLoader.discover(str(ROOT/'tests'))
stream = io.StringIO()
started = time.perf_counter()
result = unittest.TextTestRunner(stream=stream, verbosity=2).run(suite)
print(stream.getvalue())
correctness = {'architecture': platform.machine(), 'python': platform.python_version(), 'tests_run': result.testsRun, 'failures':len(result.failures), 'errors':len(result.errors), 'skipped':len(result.skipped), 'successful':result.wasSuccessful(), 'elapsed_seconds':round(time.perf_counter()-started,2), 'output':stream.getvalue()}
(ROOT/'artifacts/correctness-report.json').write_text(json.dumps(correctness,indent=2))
assert result.wasSuccessful()


/opt/venv/lib64/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


test_actual_chunked_body_limit_is_enforced (test_system.ApiTests.test_actual_chunked_body_limit_is_enforced) ... ok
test_consumer_failure_is_visible_and_does_not_deadlock_queue (test_system.ApiTests.test_consumer_failure_is_visible_and_does_not_deadlock_queue) ... ok
test_control_actions_do_not_accept_shell_commands (test_system.ApiTests.test_control_actions_do_not_accept_shell_commands) ... ok
test_event_journal_survives_reopen (test_system.ApiTests.test_event_journal_survives_reopen) ... ok
test_excessive_batch_is_rejected (test_system.ApiTests.test_excessive_batch_is_rejected) ... ok
test_external_events_cannot_impersonate_sandbox_provenance (test_system.ApiTests.test_external_events_cannot_impersonate_sandbox_provenance) ... ok
test_fractional_request_count_is_rejected (test_system.ApiTests.test_fractional_request_count_is_rejected) ... ok
test_impossible_failure_count_is_rejected (test_system.ApiTests.test_impossible_failure_count_is_rejected) ... ok
test_ingestion_is_validated_an

### Actual HTTP integration
A separate loopback app and temporary state receive synthetic fixture events via the real HTTP endpoint, bounded queue and SQLite path. The benchmark does not reset the demo app. Queue-ack timing is distinct from inference timing. No WAN or browser latency claim follows.

In [3]:
from continuityguard.http_benchmark import main as benchmark_http
http_result = benchmark_http(ROOT)
display(http_result)

{"offered_events_per_second": 10, "offered_events": 100, "queued_events": 100, "http_backpressure_events": 0, "processed_events": 100, "rejected_processing_events": 0, "duplicates": 0, "failed_batches": 0, "enqueue_seconds": 10.005, "through_final_processing_confirmation_seconds": 10.008, "processed_events_per_second": 9.99, "p95_http_queue_ack_ms": 9.332, "p95_engine_batch_ms": 16.77, "audit": {"valid": true, "records": 2, "head": "db3bc773fe7091f47c54f47f78ce8032635ad3eb2b6fe2d488d37c68ca99f1df", "scope": "local hash chain; externally retain head to detect full-history replacement"}}


{"offered_events_per_second": 50, "offered_events": 500, "queued_events": 500, "http_backpressure_events": 0, "processed_events": 500, "rejected_processing_events": 0, "duplicates": 0, "failed_batches": 0, "enqueue_seconds": 10.003, "through_final_processing_confirmation_seconds": 10.007, "processed_events_per_second": 49.97, "p95_http_queue_ack_ms": 8.272, "p95_engine_batch_ms": 8.97, "audit": {"valid": true, "records": 3, "head": "58adaf7c84da23917b492d6f845bca3154dccd067792ce2af6b0fc76a9bb43b0", "scope": "local hash chain; externally retain head to detect full-history replacement"}}


{"offered_events_per_second": 100, "offered_events": 1000, "queued_events": 1000, "http_backpressure_events": 0, "processed_events": 1000, "rejected_processing_events": 0, "duplicates": 0, "failed_batches": 0, "enqueue_seconds": 10.004, "through_final_processing_confirmation_seconds": 10.009, "processed_events_per_second": 99.91, "p95_http_queue_ack_ms": 7.982, "p95_engine_batch_ms": 9.1, "audit": {"valid": true, "records": 12, "head": "f95361617ddb8a932eadc4f11bb8b31302154a2d41f6cd65ef1c7eaafaea39d4", "scope": "local hash chain; externally retain head to detect full-history replacement"}}


{'architecture': 's390x',
 'duration_seconds_per_trial': 10,
 'batch_size': 4,
 'scope': 'Actual authenticated HTTP ingestion, bounded queue, model inference and SQLite journal on LinuxONE; controlled synthetic fixtures submitted in isolated external mode. Client and server share VM; GenAI off.',
 'latency_boundaries': {'http_queue_ack': 'Client request through validation and enqueue; excludes later inference.',
  'engine_batch': 'Feature extraction through SQLite commit; excludes queue wait.',
  'completion': 'First request through final processed-count confirmation, including polling overhead. No event end-to-end p95 or browser latency is claimed.'},
 'unauthenticated_access_rejected': True,
 'trials': [{'offered_events_per_second': 10,
   'offered_events': 100,
   'queued_events': 100,
   'http_backpressure_events': 0,
   'processed_events': 100,
   'rejected_processing_events': 0,
   'duplicates': 0,
   'failed_batches': 0,
   'enqueue_seconds': 10.005,
   'through_final_processing

### Local predictive + generative proof
The predictive model creates a sandbox incident. Local IBM Granite selects a verified explanation under a strict schema. Every evidence ID cited in its text must be included. Missing/invalid generation has a labeled deterministic fallback; this cell requires real local generation to pass.

In [4]:
from continuityguard.runtime import Engine
from continuityguard.explain import Explainer
from continuityguard.simulation import Experiment, simulate
import tempfile
events,_=simulate(Experiment('pool_exhaustion',101,strength=1.12))
with tempfile.TemporaryDirectory() as folder:
    engine=Engine(ROOT/'artifacts',str(Path(folder)/'proof.sqlite'))
    incidents=[]
    for k in range(0,len(events),4):
        incidents.extend(engine.ingest(events[k:k+4]))
        if incidents:break
    assert incidents
    incident=incidents[0]
    explainer=Explainer('/home/ibm-user/granite-4.0-h-350m-base-be.f16.gguf',engine.journal)
    explainer.request(incident)
    explainer.close()
    explanation=explainer.status(incident['id'])
    assert explanation['status']=='complete'
    assert explanation['mode']=='local Granite · constrained generation',explanation
    assert explanation['evidence_ids']==[e['id'] for e in incident['evidence']]
    proof={'architecture':platform.machine(),'model_manifest':engine.bundle.manifest,'predictive_incident':incident,'generative_explanation':explanation,'audit':engine.journal.verify(),'scope':'Actual native s390x predictive and local generative execution; synthetic incident; constrained wording, not open-ended root-cause reasoning.'}
    (ROOT/'evidence/jupyter-ai-proof.json').write_text(json.dumps(proof,indent=2))
    display(proof)
    engine.close()


llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 256


{'architecture': 's390x',
 'model_manifest': {'version': 'cg-2026-10-17-v5',
  'trained_utc': '2026-10-17T13:07:45.438758+00:00',
  'architecture': 's390x',
  'model': 'logistic_C0.1',
  'forecast_horizon_seconds': 60,
  'feature_count': 26,
  'sha256': 'ca4328b8ed2099ea4a007e75e626a4a6db63344d5c492c344733ac6f9f4d65b5',
  'packages': {'numpy': '2.0.2',
   'scikit-learn': '1.9.0',
   'scipy': '1.13.1',
   'joblib': '1.6.0'}},
 'predictive_incident': {'id': '370e309a0b34',
  'created_at': '2026-10-17T13:13:18.239385+00:00',
  'event_time': '2026-10-17T00:02:51+00:00',
  'service': 'database',
  'suspected_source': 'database',
  'category': 'pool',
  'risk': 0.9993172883987427,
  'novelty_percentile': 1.0,
  'already_breached': False,
  'affected_services': ['urgent-results', 'appointments', 'batch-reporting'],
  'evidence': [{'id': 'E3',
    'label': 'Waiting queue',
    'value': 22.2,
    'unit': 'requests'},
   {'id': 'E1',
    'label': 'Connection pool occupancy',
    'value': 90.1,
 